In [ ]:
import requests
import json
import os 
from urllib.parse import urlparse

city_name = input("enter city name :")

In [ ]:
geocoding_url="https://geocoding-api.open-meteo.com/v1/search"

params ={"name":city_name,
         "count": 3,
         "language":"en",
         "format":"json"}

response= requests.get(
    geocoding_url,
    params=params,
    timeout=30
)

response.raise_for_status()
data=response.json()
data

In [ ]:
results =data.get("results",[])

if not results:
    raise ValueError("city not found")

for index, place in enumerate(results):
    print(
        f"{index}: {place['name']},"
        f"{place.get('admin1','')},"
        f"{place.get('country','')}"
    )

In [ ]:
city_index = int(input("select city number:"))

if city_index < 0 or city_index >= len(results):
    raise ValueError("Invalid city selection!")

selected_city =results[city_index]
city = selected_city["name"]
country = selected_city ["country"]


latitude = selected_city["latitude"]
longitude = selected_city["longitude"]
timezone = selected_city["timezone"]

print("Selected city:", city)
print("Country:", country)
print("Latitude:", latitude)
print("Longitude:", longitude)
print("Timezone:", timezone)

In [ ]:
weather_url ="https://api.open-meteo.com/v1/forecast"
weather_params ={
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,relative_humidity_2m,apparent_temperature,is_day,precipitation,weather_code,wind_speed_10m",
    "timezone": timezone
}

In [ ]:
weather_response = requests .get(
    weather_url,
    params=weather_params,
    timeout=30
)

In [ ]:
weather_response.raise_for_status()
weather_json= weather_response.json()
weather_json

In [ ]:
current_weather =weather_json["current"]
current_weather

In [ ]:
temperature = current_weather["temperature_2m"]

humidity = current_weather["relative_humidity_2m"]

feels_like = current_weather["apparent_temperature"]

wind_speed = current_weather["wind_speed_10m"]

weather_code = current_weather["weather_code"]

print("Temperature:", temperature)
print("Humidity:", humidity)
print("Feels Like:", feels_like)
print("Wind Speed:", wind_speed)
print("Weather Code:", weather_code)

In [ ]:
weather_code_map = {
    0: "صاف",
    1: "عمدتاً صاف",
    2: "نیمه‌ابری",
    3: "ابری",
    45: "مه",
    48: "مه یخ‌زننده",
    51: "نم‌نم باران خفیف",
    53: "نم‌نم باران متوسط",
    55: "نم‌نم باران شدید",
    56: "باران یخ‌زننده خفیف",
    57: "باران یخ‌زننده شدید",
    61: "باران خفیف",
    63: "باران متوسط",
    65: "باران شدید",
    66: "باران یخ‌زننده خفیف",
    67: "باران یخ‌زننده شدید",
    71: "برف خفیف",
    73: "برف متوسط",
    75: "برف شدید",
    77: "دانه‌های برف",
    80: "رگبار باران خفیف",
    81: "رگبار باران متوسط",
    82: "رگبار باران شدید",
    85: "رگبار برف خفیف",
    86: "رگبار برف شدید",
    95: "رعدوبرق",
    96: "رعدوبرق همراه با تگرگ خفیف",
    99: "رعدوبرق همراه با تگرگ شدید"
}

In [ ]:
weather_code = current_weather.get("weather_code")

weather_description = weather_code_map.get(
    weather_code,
    "وضعیت هوا نامشخص است"
)

print(weather_description)

In [ ]:
weather_data = {
    "city": city,
    "country": country,
    "latitude": latitude,
    "longitude": longitude,
    "temperature": current_weather.get("temperature_2m"),
    "feels_like": current_weather.get("apparent_temperature"),
    "humidity": current_weather.get("relative_humidity_2m"),
    "wind_speed": current_weather.get("wind_speed_10m"),
    "precipitation": current_weather.get("precipitation"),
    "weather_code": weather_code,
    "weather_description": weather_description,
    "is_day": current_weather.get("is_day"),
    "time": current_weather.get("time"),
    "timezone": timezone
}

In [ ]:
weather_json_data = json.dumps(
    weather_data,
    ensure_ascii=False,
    indent=2
)

print(weather_json_data)

In [ ]:
API_KEY = os.getenv(
    "WEATHER_BOT_API_KEY",
    "MY_TEST_API_KEY_123"
)

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

In [ ]:
echo_url = "https://postman-echo.com/post"

echo_response = requests.post(
    echo_url,
    json=weather_data,
    headers=headers,
    timeout=30
)


In [ ]:
echo_response.raise_for_status()

echo_result = echo_response.json()

print("\n========== POSTMAN ECHO ==========")

print("Status Code:", echo_response.status_code)

print("\nReceived JSON:")

print(
    json.dumps(
        echo_result.get("json"),
        ensure_ascii=False,
        indent=4
    )
)

print("\nReceived Authorization:")

print(
    echo_result.get("headers", {}).get(
        "authorization",
        "Not found"
    )
)

if echo_result.get("json") == weather_data:
    print("\nPostman Echo: Data verified successfully!")
else:
    print("\nPostman Echo: Data verification failed!")

In [ ]:
import os
from getpass import getpass

os.environ["WEATHER_BOT_WEBHOOK_URL"] = getpass(
    "Enter your Webhook URL: "
)

In [ ]:
webhook_url = os.getenv("WEATHER_BOT_WEBHOOK_URL")




if not webhook_url:
    raise ValueError("Webhook URL is not configured!")

# Validate URL
parsed_url = urlparse(webhook_url)

if (
    parsed_url.scheme != "https"
    or parsed_url.hostname != "webhook.site"
    or not parsed_url.path.strip("/")
):
    raise ValueError("Invalid Webhook URL!")

# Send POST request
webhook_response = requests.post(
    webhook_url,
    json=weather_data,
    headers=headers,
    timeout=30
)

# Check response
webhook_response.raise_for_status()

print("Status Code:", webhook_response.status_code)
print("Webhook request sent successfully!")
print("Response:", webhook_response.text)